# 03 - OCR tiếng Việt và tiếng Anh (single video)

Input: output của notebook 01 (`keyframes.parquet` và `frames/`).  
Engine: PaddleOCR PP-OCRv5 Vietnamese/Latin trên GPU Kaggle.


In [ ]:
!pip uninstall -y -q paddlepaddle paddlepaddle-gpu paddleocr easyocr || true
%pip install -q paddleocr
%pip install -q paddlepaddle-gpu -i https://www.paddlepaddle.org.cn/packages/stable/cu126/


In [ ]:
from __future__ import annotations

import gc
import json
import re
import shutil
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageEnhance, ImageFilter, ImageOps
import paddle
from paddleocr import PaddleOCR

VIDEO_ID = "L21_V001"
PIPELINE_VERSION = "aicv3-ocr-single-frame-v3"
OCR_MIN_CONFIDENCE = 0.50
OCR_MAX_IMAGE_SIDE = 2200
OCR_CHECKPOINT_EVERY = 10

matches = list(Path("/kaggle/input").rglob("keyframes.parquet"))
if not matches:
    raise FileNotFoundError("Không tìm thấy keyframes.parquet từ notebook 01")
KEYFRAMES_PATH = matches[0]
INPUT_ROOT = KEYFRAMES_PATH.parent
OUTPUT_ROOT = Path("/kaggle/working/03-ocr-output-v3")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
OCR_CHECKPOINT = OUTPUT_ROOT / "ocr-checkpoint.jsonl"

keyframes_df = pd.read_parquet(KEYFRAMES_PATH).sort_values("frame_idx").reset_index(drop=True)
required_columns = {"keyframe_id", "video_id", "frame_idx", "pts_time", "frame_path"}
missing_columns = required_columns - set(keyframes_df.columns)
if missing_columns:
    raise ValueError(f"keyframes.parquet thiếu cột: {sorted(missing_columns)}")

print("Input:", INPUT_ROOT)
print("Keyframes:", len(keyframes_df))
print("Paddle version:", paddle.__version__)
print("Paddle CUDA build:", paddle.device.is_compiled_with_cuda())
print("Paddle device:", paddle.device.get_device())
if not paddle.device.is_compiled_with_cuda():
    raise RuntimeError(
        "paddlepaddle-gpu was not installed correctly. Do not continue with CPU PaddleOCR."
    )
paddle.set_device("gpu:0")


In [ ]:
def normalize_text(value: str) -> str:
    value = unicodedata.normalize("NFC", str(value))
    return re.sub(r"\s+", " ", value).strip()


def jsonable_box(box):
    points = np.asarray(box, dtype="float32").reshape(-1, 2)
    return [[float(x), float(y)] for x, y in points]


def result_payload(page):
    payload = page.json if hasattr(page, "json") else page
    if callable(payload):
        payload = payload()
    if isinstance(payload, str):
        payload = json.loads(payload)
    if isinstance(payload, dict) and isinstance(payload.get("res"), dict):
        payload = payload["res"]
    return payload if isinstance(payload, dict) else {}


def is_static_overlay(text: str) -> bool:
    compact = re.sub(r"\s+", "", text).upper()
    return bool(re.fullmatch(r"(?:HTV\w*|TV|HD|\d{1,2}:\d{2}:\d{2}|GI[AÂẠẢÃÁÀ]Y)", compact))


def extract_detections(result):
    detections = []
    for page in result:
        payload = result_payload(page)
        texts = payload.get("rec_texts", [])
        scores = payload.get("rec_scores", [])
        boxes = payload.get("rec_polys", payload.get("rec_boxes", []))
        for text, score, box in zip(texts, scores, boxes):
            text = normalize_text(text)
            confidence = float(score)
            if text and confidence >= OCR_MIN_CONFIDENCE and not is_static_overlay(text):
                detections.append({
                    "text": text,
                    "language": "vi-en-latin",
                    "confidence": confidence,
                    "bbox": jsonable_box(box),
                })
    return detections


def temporal_key(text: str) -> str:
    value = unicodedata.normalize("NFD", text.casefold())
    value = "".join(char for char in value if unicodedata.category(char) != "Mn")
    return re.sub(r"[^a-z0-9]+", "", value)


def merge_temporal_detections(detection_groups):
    merged = {}
    for detections in detection_groups:
        seen = set()
        for item in detections:
            key = temporal_key(item["text"])
            if not key or key in seen:
                continue
            seen.add(key)
            entry = merged.setdefault(key, {"support": 0, "best": item})
            entry["support"] += 1
            if item["confidence"] > entry["best"]["confidence"]:
                entry["best"] = item
    output = []
    for entry in merged.values():
        item = dict(entry["best"])
        item["temporal_support"] = entry["support"]
        output.append(item)
    return sorted(output, key=lambda item: (-item["temporal_support"], -item["confidence"]))


def resolve_frame_path(relative_path):
    path = INPUT_ROOT / relative_path
    if path.exists():
        return path
    candidates = list(INPUT_ROOT.rglob(Path(relative_path).name))
    if candidates:
        return candidates[0]
    raise FileNotFoundError(f"Frame not found: {relative_path}")


def prepare_ocr_image(path):
    image = Image.open(path).convert("RGB")
    width, height = image.size
    scale = max(1.0, OCR_MAX_IMAGE_SIDE / max(width, height))
    if scale > 1.0:
        image = image.resize((round(width * scale), round(height * scale)), Image.Resampling.LANCZOS)
    image = ImageOps.autocontrast(image, cutoff=0.5)
    image = ImageEnhance.Contrast(image).enhance(1.12)
    image = ImageEnhance.Sharpness(image).enhance(1.35)
    image = image.filter(ImageFilter.UnsharpMask(radius=1.0, percent=110, threshold=3))
    return np.asarray(image)


# Let PaddleOCR choose a compatible PP-OCRv5 detector and Vietnamese/Latin recognizer.
ocr = PaddleOCR(
    lang="vi",
    ocr_version="PP-OCRv5",
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
    device="gpu:0",
)
print("PaddleOCR PP-OCRv5 Vietnamese/Latin ready on GPU")

# Smoke test one frame before processing the full dataset.
_smoke_row = keyframes_df.iloc[0]
_smoke_path = resolve_frame_path(_smoke_row.frame_path)
try:
    _smoke_result = ocr.predict(prepare_ocr_image(_smoke_path))
    _smoke_detections = extract_detections(_smoke_result)
    print(
        "OCR smoke test passed:",
        _smoke_row.keyframe_id,
        "detections=",
        len(_smoke_detections),
    )
except Exception as exc:
    raise RuntimeError(
        "PaddleOCR GPU smoke test failed before batch processing. "
        f"{type(exc).__name__}: {exc}"
    ) from exc


In [ ]:
completed = {}
if OCR_CHECKPOINT.exists():
    for line in OCR_CHECKPOINT.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        item = json.loads(line)
        # Resume only successful records; failed records must be retried.
        if not item.get("ocr_error"):
            completed[str(item["keyframe_id"])] = item
    print("Resume successful records:", len(completed))

records = []
error_examples = []
for index, row in enumerate(keyframes_df.itertuples(index=False), start=1):
    keyframe_id = str(row.keyframe_id)
    if keyframe_id in completed:
        records.append(completed[keyframe_id])
        continue

    detections = []
    error = None
    try:
        frame_path = resolve_frame_path(row.frame_path)
        result = ocr.predict(prepare_ocr_image(frame_path))
        detections = extract_detections(result)
    except Exception as exc:
        error = f"{type(exc).__name__}: {exc}"
        if len(error_examples) < 5:
            error_examples.append({"keyframe_id": keyframe_id, "error": error})

    record = {
        "keyframe_id": keyframe_id,
        "video_id": str(row.video_id),
        "frame_idx": int(row.frame_idx),
        "pts_time": float(row.pts_time),
        "raw_text": " ".join(item["text"] for item in detections),
        "normalized_text": normalize_text(" ".join(item["text"] for item in detections)),
        "search_text": normalize_text(" ".join(item["text"] for item in detections)).casefold(),
        "text": normalize_text(" ".join(item["text"] for item in detections)),
        "detections": detections,
        "ocr_error": error,
    }
    records.append(record)
    if error is None:
        completed[keyframe_id] = record

    if index == 1 or index % OCR_CHECKPOINT_EVERY == 0 or index == len(keyframes_df):
        # Persist errors for diagnosis, but resume only successful records.
        checkpoint_records = {str(item["keyframe_id"]): item for item in records}
        with OCR_CHECKPOINT.open("w", encoding="utf-8") as file:
            for item in checkpoint_records.values():
                file.write(json.dumps(item, ensure_ascii=False) + "\n")
        print(f"OCR: {index}/{len(keyframes_df)}")
        if error_examples:
            print("Error examples:", json.dumps(error_examples, ensure_ascii=False, indent=2))

ocr_df = pd.DataFrame(records).sort_values("frame_idx").reset_index(drop=True)
error_count = int(ocr_df["ocr_error"].notna().sum())
error_rate = error_count / max(len(ocr_df), 1)
if error_rate > 0.10:
    raise RuntimeError(
        f"OCR failed for {error_count}/{len(ocr_df)} keyframes ({error_rate:.1%}). "
        f"Examples: {error_examples}"
    )

ocr_df.to_json(OUTPUT_ROOT / "ocr.jsonl", orient="records", lines=True, force_ascii=False)
ocr_parquet_df = ocr_df.copy()
ocr_parquet_df["detections_json"] = ocr_parquet_df["detections"].map(
    lambda value: json.dumps(value, ensure_ascii=False)
)
ocr_parquet_df = ocr_parquet_df.drop(columns=["detections"])
ocr_parquet_df.to_parquet(OUTPUT_ROOT / "ocr.parquet", index=False)

summary = {
    "stage": "ocr",
    "pipeline_version": PIPELINE_VERSION,
    "video_id": VIDEO_ID,
    "engine": "PaddleOCR PP-OCRv5 Vietnamese/Latin GPU",
    "languages": ["vi", "en-latin"],
    "min_confidence": OCR_MIN_CONFIDENCE,
    "max_image_side": OCR_MAX_IMAGE_SIDE,
    "frame_scope": "current-frame-only",
    "static_overlay_filter": True,
    "keyframes_processed": int(len(ocr_df)),
    "keyframes_with_text": int((ocr_df["text"].str.len() > 0).sum()),
    "detections": int(sum(len(value) for value in ocr_df["detections"])),
    "errors": error_count,
}
(OUTPUT_ROOT / "ocr-summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(json.dumps(summary, ensure_ascii=False, indent=2))


In [ ]:
# Xem nhanh các keyframe có văn bản.
display_columns = ["video_id", "frame_idx", "pts_time", "text", "ocr_error"]
display(ocr_df[ocr_df["text"].str.len() > 0][display_columns].head(30))

archive = shutil.make_archive(
    "/kaggle/working/03-ocr-L21_V001",
    "zip",
    root_dir=OUTPUT_ROOT,
)
print("Download:", archive)

del ocr
gc.collect()
try:
    paddle.device.cuda.empty_cache()
except Exception:
    pass
print("Released PaddleOCR memory")


In [ ]:
# Preview images and OCR text for the first 10 keyframes containing text.
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon

samples = ocr_df[ocr_df["text"].str.len() > 0].head(10)
if samples.empty:
    print("No keyframe contains OCR text.")
else:
    for number, row in enumerate(samples.itertuples(index=False), start=1):
        frame_path = resolve_frame_path(
            keyframes_df.loc[keyframes_df["keyframe_id"] == row.keyframe_id, "frame_path"].iloc[0]
        )
        with Image.open(frame_path) as source_image:
            image = source_image.convert("RGB")
        figure, axis = plt.subplots(figsize=(12, 7))
        axis.imshow(image)
        axis.axis("off")
        axis.set_title(
            f"{number}/10 | {row.video_id} | frame={row.frame_idx} | {row.pts_time:.2f}s\n{row.text}",
            fontsize=11,
        )
        for detection in row.detections:
            polygon = np.asarray(detection["bbox"], dtype="float32")
            axis.add_patch(
                Polygon(polygon, closed=True, fill=False, edgecolor="lime", linewidth=2)
            )
            x, y = polygon[:, 0].min(), polygon[:, 1].min()
            axis.text(
                x,
                max(0, y - 4),
                f"{detection['text']} ({detection['confidence']:.2f})",
                color="yellow",
                fontsize=10,
                bbox={"facecolor": "black", "alpha": 0.65, "pad": 2},
            )
        figure.tight_layout()
        plt.show()
        plt.close(figure)
